# DistilBERT Exploration
This notebook explores the DistilBERT model from HuggingFace — the backbone used inside our Q-Former implementation.

## 1. Load the model

In [16]:
import transformers
import torch
print(transformers.__version__)

5.4.0


In [2]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

c:\Users\raoad\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.0
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
model = AutoModelForMaskedLM.from_pretrained("distilbert/distilbert-base-uncased")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

c:\Users\raoad\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\raoad\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## 2. Model architecture
Print the full module tree so you can see every sub-module and how they nest.

In [4]:
print(model)

DistilBertForMaskedLM(
  (activation): GELUActivation()
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0

## 3. Parameter count

In [5]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total:,}')
print(f'Trainable parameters: {trainable:,}')

Total parameters    : 66,985,530
Trainable parameters: 66,985,530


## 4. Inspect the embeddings layer
This is what Q-Former extracts via `bert_model.embeddings` — converts token IDs to vectors.

In [11]:
embeddings = model.distilbert.embeddings
print(embeddings)
print()
print('word_embeddings  shape:', embeddings.word_embeddings.weight.shape)   # (vocab_size, hidden)
print('position_embeddings shape:', embeddings.position_embeddings.weight.shape)  # (max_pos, hidden)

Embeddings(
  (word_embeddings): Embedding(30522, 768, padding_idx=0)
  (position_embeddings): Embedding(512, 768)
  (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

word_embeddings  shape: torch.Size([30522, 768])
position_embeddings shape: torch.Size([512, 768])


## 5. Inspect the transformer layers
Q-Former extracts these via `bert_model.transformer.layer` — there are 6 of them.

In [12]:
layers = model.distilbert.transformer.layer
print(f'Number of transformer layers: {len(layers)}')
print()
print('Single layer structure:')
print(layers[0])

Number of transformer layers: 6

Single layer structure:
TransformerBlock(
  (attention): DistilBertSelfAttention(
    (q_lin): Linear(in_features=768, out_features=768, bias=True)
    (k_lin): Linear(in_features=768, out_features=768, bias=True)
    (v_lin): Linear(in_features=768, out_features=768, bias=True)
    (out_lin): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (ffn): FFN(
    (dropout): Dropout(p=0.1, inplace=False)
    (lin1): Linear(in_features=768, out_features=3072, bias=True)
    (lin2): Linear(in_features=3072, out_features=768, bias=True)
    (activation): GELUActivation()
  )
  (output_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
)


## 6. Inside one transformer layer
Each layer has: MultiHeadSelfAttention → sa_layer_norm → FFN → output_layer_norm

In [13]:
layer0 = layers[0]

attn = layer0.attention
print('--- Attention ---')
print(f'  n_heads   : {attn.n_heads}')
print(f'  q_lin weight shape: {attn.q_lin.weight.shape}')  # (hidden, hidden)
print(f'  k_lin weight shape: {attn.k_lin.weight.shape}')
print(f'  v_lin weight shape: {attn.v_lin.weight.shape}')
print(f'  out_lin weight shape: {attn.out_lin.weight.shape}')

print()
print('--- FFN ---')
ffn = layer0.ffn
print(f'  lin1 weight shape: {ffn.lin1.weight.shape}')  # (4*hidden, hidden)
print(f'  lin2 weight shape: {ffn.lin2.weight.shape}')  # (hidden, 4*hidden)

--- Attention ---
  n_heads   : 12
  q_lin weight shape: torch.Size([768, 768])
  k_lin weight shape: torch.Size([768, 768])
  v_lin weight shape: torch.Size([768, 768])
  out_lin weight shape: torch.Size([768, 768])

--- FFN ---
  lin1 weight shape: torch.Size([3072, 768])
  lin2 weight shape: torch.Size([768, 3072])


## 7. Tokenize a sentence and inspect token IDs

In [14]:
sentence = "A cat is sitting on a mat."
tokens = tokenizer(sentence, return_tensors='pt')

print('input_ids     :', tokens['input_ids'])
print('attention_mask:', tokens['attention_mask'])
print()
print('Decoded tokens:', tokenizer.convert_ids_to_tokens(tokens['input_ids'][0].tolist()))

input_ids     : tensor([[  101,  1037,  4937,  2003,  3564,  2006,  1037, 13523,  1012,   102]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Decoded tokens: ['[CLS]', 'a', 'cat', 'is', 'sitting', 'on', 'a', 'mat', '.', '[SEP]']


## 8. Full forward pass — shapes at each stage

In [25]:
with torch.no_grad():
    output = model(**tokens, output_hidden_states=True)

print('Output keys:', output.keys())
for key in output.keys():
    print(f'  {key}')
print('last_hidden_state shape:', output.hidden_states[-1].shape)  # (B, T, 768)
print()
print('Hidden states at every layer (including embedding layer):')
for i, hs in enumerate(output.hidden_states):
    label = 'embeddings' if i == 0 else f'layer {i}'
    print(f'  {label:12s}: {hs.shape}')

Output keys: odict_keys(['logits', 'hidden_states'])
  logits
  hidden_states
last_hidden_state shape: torch.Size([1, 10, 768])

Hidden states at every layer (including embedding layer):
  embeddings  : torch.Size([1, 10, 768])
  layer 1     : torch.Size([1, 10, 768])
  layer 2     : torch.Size([1, 10, 768])
  layer 3     : torch.Size([1, 10, 768])
  layer 4     : torch.Size([1, 10, 768])
  layer 5     : torch.Size([1, 10, 768])
  layer 6     : torch.Size([1, 10, 768])


## 9. Embeddings layer in isolation
This is exactly what Q-Former calls: `self.embeddings(input_ids=text_input_ids)`

In [26]:
with torch.no_grad():
    emb_out = model.distilbert.embeddings(input_ids=tokens['input_ids'])

print('Embeddings output shape:', emb_out.shape)  # (B, T, 768)
print('This is the input to the first transformer layer.')

Embeddings output shape: torch.Size([1, 10, 768])
This is the input to the first transformer layer.


## 10. Single transformer layer in isolation
Q-Former calls each layer individually: `layer(x, attention_mask)[0]`

In [27]:
with torch.no_grad():
    layer_out = layers[0](emb_out)   # returns a tuple

print('Return type :', type(layer_out))
print('Tuple length:', len(layer_out))
print('layer_out[0] shape:', layer_out[0].shape)  # (B, T, 768) — the hidden states

Return type : <class 'torch.Tensor'>
Tuple length: 1
layer_out[0] shape: torch.Size([10, 768])


## 11. Attention mask format DistilBERT expects
DistilBERT's transformer layers accept a `(B, T, T)` boolean mask (NOT `(B, 1, T, T)` like standard BERT).

In [28]:
B, T, H = emb_out.shape

# Full attention mask — every token attends to every other token
full_mask = torch.ones(B, T, T, dtype=torch.bool)

# Causal mask — each token only attends to itself and earlier tokens
causal_mask = torch.tril(torch.ones(B, T, T, dtype=torch.bool))

with torch.no_grad():
    out_full   = layers[0](emb_out, attn_mask=full_mask)[0]
    out_causal = layers[0](emb_out, attn_mask=causal_mask)[0]

print('Full mask output shape  :', out_full.shape)
print('Causal mask output shape:', out_causal.shape)
print()
print('Outputs differ (causal != full):', not torch.allclose(out_full, out_causal))

Full mask output shape  : torch.Size([10, 768])
Causal mask output shape: torch.Size([10, 768])

Outputs differ (causal != full): False


## 12. [CLS] token
Position 0 is always the `[CLS]` token — Q-Former uses `txt_emb[:, 0]` as the pooled text representation.

In [33]:
with torch.no_grad():
    out = model(**tokens)

cls_token = out.last_hidden_state[:, 0]   # position 0 = [CLS], shape (B, 768)
print('[CLS] token vector shape:', cls_token.shape)
print('[CLS] token ID in vocab :', tokenizer.cls_token_id, '→', tokenizer.cls_token)

AttributeError: 'MaskedLMOutput' object has no attribute 'last_hidden_state'

## 13. Cosine similarity between two sentences
Demonstrates how [CLS] embeddings can measure sentence similarity — the same idea behind Q-Former's ITC loss.

In [34]:
import torch.nn.functional as F

sentences = [
    "A cat is sitting on a mat.",
    "A kitten is resting on a rug.",
    "The stock market crashed today.",
]

embeddings_list = []
for s in sentences:
    t = tokenizer(s, return_tensors='pt')
    with torch.no_grad():
        emb = model(**t).last_hidden_state[:, 0, :]   # [CLS]
    embeddings_list.append(F.normalize(emb, dim=-1))

for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = (embeddings_list[i] * embeddings_list[j]).sum().item()
        print(f'sim("{sentences[i][:30]}...", "{sentences[j][:30]}...") = {sim:.4f}')

AttributeError: 'MaskedLMOutput' object has no attribute 'last_hidden_state'